# Configuration Space: Visualizing C-Space for a 2-DOF Arm

## What is C-Space and why do we plan in it?

The **configuration space** (C-space) of a robot is the set of all possible joint angle combinations. For a 2-DOF planar arm, each point `(θ1, θ2)` in C-space *completely* describes the robot's pose — there is no ambiguity.

Planning in task space (Cartesian XYZ) sounds intuitive, but it quickly breaks down:
- Multiple joint configurations can produce the same EE position (kinematic redundancy)
- Straight lines in task space may require non-straight paths in joint space
- Obstacles defined in task space become complex, curved regions in joint space

**C-space planning solves this cleanly.** Every robot state is a single point. A motion plan is a path between two such points. Collision with an obstacle is just asking: "Is this C-space point forbidden?"

Let's build the intuition visually.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

In [ ]:
# Define a simple 2-DOF planar arm
L1 = 1.0  # length of link 1
L2 = 0.8  # length of link 2

def fk_2dof(theta1, theta2):
    """Forward kinematics for a 2-DOF planar arm.
    
    Returns (x, y) position of the end-effector.
    theta1: rotation of link 1 about base
    theta2: rotation of link 2 relative to link 1
    """
    x = L1 * np.cos(theta1) + L2 * np.cos(theta1 + theta2)
    y = L1 * np.sin(theta1) + L2 * np.sin(theta1 + theta2)
    return x, y

# Quick sanity check
x, y = fk_2dof(0.0, 0.0)  # fully extended along X-axis
print(f"FK at (0, 0): EE position = ({x:.3f}, {y:.3f})")
print(f"Expected: ({L1 + L2:.3f}, 0.000)  — fully extended")

## Sweeping C-Space

Now let's systematically sample the entire C-space. For each pair `(θ1, θ2)` in `[-π, π] × [-π, π]`, we compute the end-effector position using FK.

This is the **workspace** of the arm — all EE positions reachable by some joint configuration. Notice that the same XY point may be reachable by multiple `(θ1, θ2)` pairs (redundancy), and some XY regions are completely unreachable.

In [ ]:
N = 50  # grid resolution per axis

theta1_vals = np.linspace(-np.pi, np.pi, N)
theta2_vals = np.linspace(-np.pi, np.pi, N)

# Create meshgrid and compute FK for all combinations
T1, T2 = np.meshgrid(theta1_vals, theta2_vals)
EE_x, EE_y = fk_2dof(T1, T2)

print(f"C-space grid: {N}x{N} = {N*N} configurations")
print(f"EE x-range: [{EE_x.min():.3f}, {EE_x.max():.3f}]")
print(f"EE y-range: [{EE_y.min():.3f}, {EE_y.max():.3f}]")
print(f"Max reach: L1 + L2 = {L1 + L2:.3f}  (matches max of |EE|)")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

# Scatter: each point is the EE position for one (θ1, θ2) pair
# Color by θ1 so we can see which base rotation produces which EE position
sc = ax.scatter(
    EE_x.ravel(), EE_y.ravel(),
    c=T1.ravel(), cmap='hsv', s=4, alpha=0.6
)

cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('θ1 (base joint angle) [rad]')

# Annotate unreachable inner circle
inner_radius = abs(L1 - L2)
outer_radius = L1 + L2
circle_inner = plt.Circle((0, 0), inner_radius, fill=False, color='k', linestyle='--', linewidth=1.5)
circle_outer = plt.Circle((0, 0), outer_radius, fill=False, color='k', linestyle='--', linewidth=1.5)
ax.add_patch(circle_inner)
ax.add_patch(circle_outer)
ax.text(inner_radius + 0.05, 0.05, 'inner limit', fontsize=8)
ax.text(outer_radius - 0.4, 0.05, 'outer limit', fontsize=8)

ax.set_title('Task space coverage from C-space sweep', fontsize=13)
ax.set_xlabel('x [m]')
ax.set_ylabel('y [m]')
ax.set_aspect('equal')
ax.axhline(0, color='gray', linewidth=0.5)
ax.axvline(0, color='gray', linewidth=0.5)
plt.tight_layout()
plt.show()

print("Notice how the same color (same θ1) traces arcs — the second joint sweeps along each arc.")

## Adding an Obstacle in Task Space

Now suppose there is a circular obstacle at position `[1.2, 0.5]` with radius `0.3`. Any end-effector position inside this circle is forbidden.

The key question is: **which joint configurations `(θ1, θ2)` produce an EE position inside the obstacle?** Those are the *forbidden configurations* in C-space.

We'll compute this by checking every grid point against the obstacle.

In [ ]:
# Define circular obstacle in task space
obstacle_center = np.array([1.2, 0.5])
obstacle_radius = 0.3

# Check which (θ1, θ2) configurations produce an EE inside the obstacle
dist_to_obstacle = np.sqrt((EE_x - obstacle_center[0])**2 + (EE_y - obstacle_center[1])**2)
in_collision = dist_to_obstacle < obstacle_radius

n_colliding = in_collision.sum()
n_total = N * N
print(f"Obstacle at {obstacle_center}, radius {obstacle_radius}")
print(f"Colliding C-space configurations: {n_colliding} / {n_total}  ({100*n_colliding/n_total:.1f}%)")
print()
print("A simple disk in task space maps to a more complex region in C-space (next plot).")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Left: task space with obstacle ---
ax = axes[0]
free_mask = ~in_collision
ax.scatter(EE_x[free_mask], EE_y[free_mask], c='steelblue', s=4, alpha=0.5, label='Free')
ax.scatter(EE_x[in_collision], EE_y[in_collision], c='red', s=6, alpha=0.8, label='In obstacle')
obs_circle = plt.Circle(obstacle_center, obstacle_radius, fill=True, facecolor='red', alpha=0.3,
                         edgecolor='darkred', linewidth=2)
ax.add_patch(obs_circle)
ax.set_title('Task space — red disk = obstacle', fontsize=12)
ax.set_xlabel('x [m]')
ax.set_ylabel('y [m]')
ax.set_aspect('equal')
ax.legend(markerscale=3, loc='upper left')
ax.set_xlim(-2, 2)
ax.set_ylim(-2, 2)

# --- Right: C-space heatmap ---
ax2 = axes[1]
# Color: 0 = free (blue), 1 = forbidden (red)
cmap = mcolors.ListedColormap(['steelblue', 'red'])
im = ax2.imshow(
    in_collision.astype(float),
    origin='lower',
    extent=[-np.pi, np.pi, -np.pi, np.pi],
    cmap=cmap,
    vmin=0, vmax=1,
    aspect='auto',
    alpha=0.8
)
ax2.set_title('C-space with obstacle (red = forbidden)', fontsize=12)
ax2.set_xlabel('θ1 [rad]')
ax2.set_ylabel('θ2 [rad]')
cbar2 = plt.colorbar(im, ax=ax2, ticks=[0.25, 0.75])
cbar2.ax.set_yticklabels(['Free', 'Forbidden'])

plt.suptitle('From task space obstacle to C-space forbidden region', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## Key Insight

Look at the C-space plot on the right. The obstacle was a **simple disk** in task space — a geometrically trivial shape. Yet in C-space it becomes a **curved, irregular region** that wraps around.

This is the central challenge of motion planning:

1. **Obstacle shapes are not preserved** when mapped from task space to C-space. Simple task-space obstacles become complex C-space regions.

2. **Sampling-based planners** (RRT, PRM) work directly in C-space without ever explicitly computing this map. They just check: "is this configuration valid?" — which is fast and avoids building the C-space obstacle map explicitly.

3. **The free space is connected** (most of the time), but the forbidden region may disconnect it into multiple components. A path from A to B might need to go around a C-space obstacle that has nothing to do with the task-space shape.

As DOF increases from 2 to 6 (like the UR5), this C-space lives in 6 dimensions — we can no longer visualize it, which is exactly why we need smarter search algorithms.

In [ ]:
print("Exercise: Try moving the obstacle to [0.5, 1.0]. How does the forbidden region change?")
print()
print("To try it:")
print("  1. Change obstacle_center to np.array([0.5, 1.0]) in the 'collision-check' cell above")
print("  2. Re-run that cell and the plot cell")
print("  3. Does the forbidden region become larger or smaller? More or less symmetric?")
print()
print("Bonus: What happens when the obstacle is at [0.0, 0.0] (the base)?")
print("       What about radius=1.5 (a very large obstacle)?")